# notebook for trying a few ideas

In [ ]:
import networkx as nx
import numpy as np
from GraphRicciCurvature.OllivierRicci import OllivierRicci
from GraphRicciCurvature.FormanRicci import FormanRicci


def degree_activation(threshold, graph):
    degrees = dict(graph.degree())
    return {node for node, degree in degrees.items() if degree <= threshold}


def closeness_activation(threshold, graph):
    closeness = nx.closeness_centrality(graph)
    return {node for node, centrality in closeness.items() if centrality <= threshold}

def betweenness_activation(threshold, graph):
    betweenness = nx.betweenness_centrality(graph)
    return {node for node, centrality in betweenness.items() if centrality <= threshold}


def weight_activation(threshold, graph):
    return {node for node in graph.nodes if sum(data.get('value', 1) for _, _, data in graph.edges(node, data=True)) <= threshold} 


def degree_centrality_activation(threshold, graph):
    degree_centrality = nx.degree_centrality(graph)
    return {node for node, centrality in degree_centrality.items() if centrality <= threshold}


def forman_ricci_activation(threshold, graph):
    forman_ricci = FormanRicci(graph)
    forman_ricci.compute_ricci_curvature()

    # Extract curvature values for nodes
    node_curvatures = nx.get_node_attributes(graph, "formanCurvature")
    
    # Return nodes where curvature is less than or equal to the threshold
    return {node for node, curvature in node_curvatures.items() if curvature <= threshold}


def ollivier_ricci_activation(threshold, graph):
    ollivier_ricci = OllivierRicci(graph)
    ollivier_ricci.compute_ricci_curvature()

    # Extract curvature values for nodes
    return {node for node, curvature in ollivier_ricci.G.nodes(data="ricciCurvature") if curvature >= threshold}


def compute_degree_thresholds(graphs, num_buckets=10):
    all_degrees = []
    
    # Collect all degrees from all graphs
    for graph in graphs:
        degrees = dict(graph.degree()).values()
        all_degrees.extend(degrees)
    
    # Compute the thresholds based on percentiles of the degree distribution
    thresholds = np.percentile(all_degrees, np.linspace(0, 100, num_buckets + 1)[1:])
    
    return thresholds


def compute_closeness_thresholds(graphs, num_buckets=10):
    all_closeness = []
    
    # Collect all closeness centrality values from all graphs
    for graph in graphs:
        closeness = nx.closeness_centrality(graph)
        all_closeness.extend(closeness.values())  # Collect centrality values as a list
    
    # Compute the thresholds based on percentiles of the closeness centrality distribution
    thresholds = np.percentile(all_closeness, np.linspace(0, 100, num_buckets + 1)[1:])
    
    return thresholds


def compute_betweenness_centrality_thresholds(graphs, num_buckets=10):
    all_betweenness_centralities = []
    
    # Collect all betweenness centrality values from all graphs
    for graph in graphs:
        # Compute betweenness centrality for all nodes in the graph
        betweenness = nx.betweenness_centrality(graph)
        
        # Collect the betweenness centrality values
        all_betweenness_centralities.extend(betweenness.values())
    
    # Compute the thresholds based on percentiles of the betweenness centrality distribution
    thresholds = np.percentile(all_betweenness_centralities, np.linspace(0, 100, num_buckets + 1)[1:])
    
    return thresholds


def compute_degree_centrality_thresholds(graphs, num_buckets=10):
    all_degree_centrality = []
    
    # Collect all degree centrality values from all graphs
    for graph in graphs:
        degree_centrality = nx.degree_centrality(graph)
        all_degree_centrality.extend(degree_centrality.values())  # Collect centrality values as a list
    
    # Compute the thresholds based on percentiles of the degree centrality distribution
    thresholds = np.percentile(all_degree_centrality, np.linspace(0, 100, num_buckets + 1)[1:])
    
    return thresholds


def compute_weight_thresholds(graphs, num_buckets=10):
    all_weighted_degrees = []
    
    # Collect all weighted degrees from all graphs
    for graph in graphs:
        weighted_degrees = []
        
        if graph.is_directed():
            # For directed graphs, sum incoming and outgoing edge values
            for node in graph.nodes():
                in_value = sum(value for _, _, value in graph.in_edges(node, data='value'))
                out_value = sum(value for _, _, value in graph.out_edges(node, data='value'))
                weighted_degrees.append(in_value + out_value)
        else:
            # For undirected graphs, sum values of all edges
            for node in graph.nodes():
                weighted_degree = sum(value for _, _, value in graph.edges(node, data='value'))
                weighted_degrees.append(weighted_degree)
        
        all_weighted_degrees.extend(weighted_degrees)
    
    # Compute the thresholds based on percentiles of the weighted degree distribution
    thresholds = np.percentile(all_weighted_degrees, np.linspace(0, 100, num_buckets + 1)[1:])
    
    return thresholds


def compute_forman_ricci_thresholds(graphs, num_buckets=10):
    all_forman_ricci_curvatures = []
    
    # Collect all Forman-Ricci curvatures from all graphs
    for graph in graphs:
        edWts = nx.get_edge_attributes(graph,'value')
        formanRicci = {}
        for ed in graph.edges():
            # See calc at   https://doi.org/10.1016/j.chaos.2018.11.031
            v1 = ed[0]
            v2 = ed[1]
            #Assuming all vertex weights are 1 since unspecified
            wtv1 = 1.0
            wtv2 = 1.0
            we = edWts[ed] # weight of edge

            sumV1I = 0.0
            sumV2O = 0.0
            for v1pred in graph.predecessors(v1):
                sumV1I += wtv1/np.sqrt(edWts[(v1pred,v1)] * we)
            for v2nxt in graph.successors(v2):
                sumV2O += wtv2/np.sqrt(edWts[(v2,v2nxt)] * we)
            # check if bidirection, then would need to consider V1 out and V2 int as well
            if any(temp == v2 for temp in graph.predecessors(v1)):
                for v1nxt in graph.successors(v1):
                    sumV1I += wtv1/np.sqrt(edWts[(v1,v1nxt)] * we)
                for v2pred in graph.predecessors(v2):
                    sumV2O += wtv2/np.sqrt(edWts[(v2pred,v2)] * we)
            formanRicci[ed] = np.round(we * (wtv1/we - sumV1I) + we * (wtv2/we - sumV2O), 3)
            
            # Compute the thresholds based on percentiles of the Forman-Ricci curvature distribution
            thresholds = np.percentile(all_forman_ricci_curvatures, np.linspace(0, 100, num_buckets + 1)[1:])
            
            return thresholds


from collections import Counter
# Helper function
def wasserstein_distance(neighbors_u, neighbors_v):
    """
    Compute the Wasserstein distance between two sets of neighbors.
    The sets are assumed to be collections of neighboring nodes of u and v.
    """
    # Count the frequency of neighbors for both u and v
    counter_u = Counter(neighbors_u)
    counter_v = Counter(neighbors_v)
    
    # Combine all unique neighbors
    all_neighbors = set(neighbors_u) | set(neighbors_v)
    
    # Compute the Wasserstein distance as the sum of absolute differences of frequencies
    distance = 0
    for neighbor in all_neighbors:
        distance += abs(counter_u.get(neighbor, 0) - counter_v.get(neighbor, 0))
    
    return distance


def compute_ollivier_ricci_thresholds(graphs, num_buckets=10):
    all_ollivier_ricci_curvatures = []
    
    # Collect all Ollivier-Ricci curvatures from all graphs
    for graph in graphs:
        ollivier_ricci_curvatures = []
        
        # For each edge in the graph, calculate the Ollivier-Ricci curvature
        for u, v in graph.edges():
            neighbors_u = list(graph.neighbors(u))
            neighbors_v = list(graph.neighbors(v))
            
            # Calculate the Wasserstein distance between the neighbors of u and v
            W_uv = wasserstein_distance(neighbors_u, neighbors_v)
            
            # Ollivier-Ricci curvature calculation
            degree_u = graph.degree(u)
            degree_v = graph.degree(v)
            
            curvature = 1 - (W_uv / max(degree_u, degree_v))
            ollivier_ricci_curvatures.append(curvature)
        
        all_ollivier_ricci_curvatures.extend(ollivier_ricci_curvatures)
    
    # Compute the thresholds based on percentiles of the Ollivier-Ricci curvature distribution
    thresholds = np.percentile(all_ollivier_ricci_curvatures, np.linspace(0, 100, num_buckets + 1)[1:])
    
    return thresholds

def graph_to_undirected(graph):
    undirected_graph = nx.Graph()
    for u, v, data in graph.edges(data=True):
        if undirected_graph.has_edge(u, v):
            # Combine the values, e.g., take the max
            undirected_graph[u][v]['value'] = max(undirected_graph[u][v]['value'], data['value'])
        else:
            undirected_graph.add_edge(u, v, **data)

    # Inspect the undirected graph
    return undirected_graph


def process_graphs_for_embeddings(graphs, thresholds_func, activation_func, num_buckets=10):
    all_embeddings = []
    
    # Compute global thresholds using the linear bucketing method
    thresholds = thresholds_func(graphs, num_buckets=num_buckets)
    
    for graph in graphs: 
        active_data = graph_filtration(graph,  activation_func, thresholds)   
        all_embeddings.append(active_data)  # 20-dimensional embedding
    
    return all_embeddings


# Filtration function for generating embeddings (50-dimensional active data)
def graph_filtration(graph, activation_func, thresholds):
    active_data = []

    for threshold in thresholds:
        # Get active node set based on the activation function
        active_node_set = activation_func(threshold, graph)
        
        # Filter edges to include only those between active nodes
        active_edges = {edge for edge in graph.edges(data=False) 
                        if edge[0] in active_node_set and edge[1] in active_node_set}
        
        # Compute features:
        node_count = len(active_node_set)  # Number of active nodes
        
        # In-degree: count edges directed towards active nodes (edges where the target is active)
        in_degree = sum(1 for edge in active_edges if edge[1] in active_node_set and edge[0] in active_node_set)
        
        # Out-degree: count edges directed from active nodes (edges where the source is active)
        out_degree = sum(1 for edge in active_edges if edge[0] in active_node_set and edge[1] in active_node_set)
        
        # Weight calculations:
        # In-weight: sum the weights of edges directed towards active nodes (edges where target is active)
        in_weight = sum(graph[edge[0]][edge[1]].get('value', 1) for edge in active_edges if edge[1] in active_node_set and edge[0] in active_node_set)
        
        # Out-weight: sum the weights of edges directed from active nodes (edges where source is active)
        out_weight = sum(graph[edge[0]][edge[1]].get('value', 1) for edge in active_edges if edge[0] in active_node_set and edge[1] in active_node_set)
        
        # Append the features for this threshold
        active_data.extend([node_count, in_degree, out_degree, in_weight, out_weight])

    return active_data  # Returns a vector with features for all thresholds

In [3]:
from utils.visualizers import Visualizer
from utils.loader import Loader

dataset = 'networkaragon'

my_loader = Loader()
my_visualizer = Visualizer(dataset, 'regression')

data, labels = my_loader.load_data(dataset)
'''   
# Testing
print("Activation by Degree; 10 thresholds")
embeddings = process_graphs_for_embeddings(data, compute_degree_thresholds, degree_activation, num_buckets=10)
print(data[0])
print(embeddings[0])
my_visualizer.display_single_embedding(embeddings[0], num_buckets=10)

print("Activation by Degree; 15 thresholds")
embeddings = process_graphs_for_embeddings(data, compute_degree_thresholds, degree_activation, num_buckets=15)
print(data[0])
print(embeddings[0])
my_visualizer.display_single_embedding(embeddings[0], num_buckets=15)

print("Activation by Closeness; 10 thresholds")
embeddings = process_graphs_for_embeddings(data, compute_closeness_thresholds, closeness_activation, num_buckets=10)
print(data[0])
print(embeddings[0])
my_visualizer.display_single_embedding(embeddings[0], num_buckets=10)

print("Activation by Closeness; 15 thresholds")
embeddings = process_graphs_for_embeddings(data, compute_closeness_thresholds, closeness_activation, num_buckets=15)
print(data[0])
print(embeddings[0])
my_visualizer.display_single_embedding(embeddings[0], num_buckets=15)

print("Activation by Betweenness; 10 thresholds")
embeddings = process_graphs_for_embeddings(data, compute_betweenness_centrality_thresholds, betweenness_activation, num_buckets=10)
print(data[0])
print(embeddings[0])
my_visualizer.display_single_embedding(embeddings[0], num_buckets=10)

print("Activation by Betweenness; 15 thresholds")
embeddings = process_graphs_for_embeddings(data, compute_betweenness_centrality_thresholds, betweenness_activation, num_buckets=15)
print(data[0])
print(embeddings[0])
my_visualizer.display_single_embedding(embeddings[0], num_buckets=15)

print("Activation by Degree Centrality; 10 thresholds")
embeddings = process_graphs_for_embeddings(data, compute_degree_centrality_thresholds, degree_centrality_activation, num_buckets=10)
print(data[0])
print(embeddings[0])
my_visualizer.display_single_embedding(embeddings[0], num_buckets=10)

print("Activation by Degree Centrality; 15 thresholds")
embeddings = process_graphs_for_embeddings(data, compute_degree_centrality_thresholds, degree_centrality_activation, num_buckets=15)
print(data[0])
print(embeddings[0])
my_visualizer.display_single_embedding(embeddings[0], num_buckets=15)

print("Activation by Weight; 10 thresholds")
embeddings = process_graphs_for_embeddings(data, compute_weight_thresholds, weight_activation, num_buckets=10)
print(data[0])
print(embeddings[0])
my_visualizer.display_single_embedding(embeddings[0], num_buckets=10)

print("Activation by Weight; 15 thresholds")
embeddings = process_graphs_for_embeddings(data, compute_weight_thresholds, weight_activation, num_buckets=15)
print(data[0])
print(embeddings[0])
my_visualizer.display_single_embedding(embeddings[0], num_buckets=15)'''

print("Activation by O. Ricci; 10 thresholds")
undirected_graphs = []
for graph in data:
    undirected_graphs.append(graph_to_undirected(graph))
embeddings = process_graphs_for_embeddings(undirected_graphs, compute_ollivier_ricci_thresholds, ollivier_ricci_activation, num_buckets=10)
print(undirected_graphs[0])
print(embeddings[0])
my_visualizer.display_single_embedding(embeddings[0], num_buckets=10)

print("Activation by O. Ricci; 15 thresholds")
undirected_graphs = []
for graph in data:
    undirected_graphs.append(graph_to_undirected(graph))
embeddings = process_graphs_for_embeddings(undirected_graphs, compute_ollivier_ricci_thresholds, ollivier_ricci_activation, num_buckets=10)
print(undirected_graphs[0])
print(embeddings[0])
my_visualizer.display_single_embedding(embeddings[0], num_buckets=15)

print("Activation by F. Ricci; 10 thresholds")
undirected_graphs = []
for graph in data:
    undirected_graphs.append(graph_to_undirected(graph))
embeddings = process_graphs_for_embeddings(undirected_graphs, compute_forman_ricci_thresholds, forman_ricci_activation, num_buckets=10)
print(undirected_graphs[0])
print(embeddings[0])
my_visualizer.display_single_embedding(embeddings[0], num_buckets=10)

print("Activation by F. Ricci; 15 thresholds")
undirected_graphs = []
for graph in data:
    undirected_graphs.append(graph_to_undirected(graph))
embeddings = process_graphs_for_embeddings(undirected_graphs, compute_forman_ricci_thresholds, forman_ricci_activation, num_buckets=10)
print(undirected_graphs[0])
print(embeddings[0])
my_visualizer.display_single_embedding(embeddings[0], num_buckets=15)




Activation by O. Ricci; 10 thresholds


ValueError: cannot find context for 'fork'

In [ ]:
import numpy as np
import networkx as nx
import random
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn

# Function to generate graphs with a cosine cycle pattern for nodes with random edges
def generate_cosine_pattern_graphs(num_graphs, max_nodes, avg_edges_per_node, period_days, start_size):
    graphs = []
    
    # Cosine pattern with a cycle (adjusted period for nodes)
    node_pattern = (max_nodes / 2) * (1 + np.cos(2 * np.pi * np.arange(num_graphs) / period_days))

    # Create graphs based on the node pattern
    for i in range(num_graphs):
        # Add more randomness to the number of nodes around the cosine pattern to increase variation
        num_nodes = start_size + max(2, min(int(node_pattern[i] + random.uniform(-0.3 * max_nodes, 0.3 * max_nodes)), max_nodes))

        # Create an empty graph with num_nodes
        G = nx.empty_graph(num_nodes)

        # Ensure each node has at least one edge
        for node in G.nodes():
            possible_edges = [edge for edge in nx.non_edges(G) if node in edge]
            if possible_edges:
                selected_edge = random.choice(possible_edges)
                G.add_edge(*selected_edge)
        
        # Randomly add additional edges based on the average number of edges per node
        num_edges = avg_edges_per_node * num_nodes
        possible_edges = list(nx.non_edges(G))
        if num_edges > len(possible_edges):
            num_edges = len(possible_edges)
        selected_edges = np.random.choice(len(possible_edges), int(num_edges), replace=False)
        G.add_edges_from([possible_edges[j] for j in selected_edges])

        # Print the graph details for verification
        print(f"Graph {i}: Nodes = {G.number_of_nodes()}, Edges = {G.number_of_edges()}")
        graphs.append(G)

    return graphs

# Parameters for graph generation
num_graphs = 100 # Number of graphs for training
max_nodes = 100  # Maximum number of nodes
avg_edges_per_node = 10  # Average number of edges per node
period_days = 30  # Set the period of the cosine cycle
start_size = 10  # Starting size for nodes

# Generate graphs with cosine patterns for nodes and random edges
synthetic_graphs = generate_cosine_pattern_graphs(num_graphs, max_nodes, avg_edges_per_node, period_days, start_size)

# Optional: Plot the number of nodes over time
node_pattern = (max_nodes / 2) * (1 + np.cos(2 * np.pi * np.arange(num_graphs) / period_days))
num_nodes_with_randomness = [
    start_size + max(2, min(int(n + np.random.uniform(-0.3 * max_nodes, 0.3 * max_nodes)), max_nodes))
    for n in node_pattern
]

plt.figure(figsize=(10, 6))
plt.plot(num_nodes_with_randomness, label="Number of Nodes", color="blue")
plt.title(f"Number of Nodes Over Time with {period_days}-Day Cosine Pattern")
plt.xlabel("Time (Graph Index)")
plt.ylabel("Number of Nodes")
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
import numpy as np
import networkx as nx
import random
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn

# Node activation functions for degree-based filtration
def degree_activation(degrees, threshold, last_threshold=False):
    if last_threshold:
        return {node for node, degree in degrees.items()}
    else:
        return {node for node, degree in degrees.items() if degree <= threshold}

# Compute global degree thresholds based on the degree distribution across all graphs
def compute_degree_thresholds(graphs, num_thresholds=10):
    all_degrees = []
    
    # Collect all degrees from all graphs
    for graph in graphs:
        degrees = dict(graph.degree()).values()
        all_degrees.extend(degrees)
    
    # Compute the thresholds based on quantiles of the degree distribution
    thresholds = np.percentile(all_degrees, np.linspace(0, 100, num_thresholds + 1)[1:])
    
    return thresholds

# Filtration function for generating embeddings (20-dimensional active data)
def graph_filtration(graph, metric, activation_func, thresholds):
    active_data = []
    
    for i, threshold in enumerate(thresholds):
        last_threshold = (i == len(thresholds) - 1)
        active_node_set = activation_func(metric, threshold, last_threshold)
        active_edges = {edge for edge in graph.edges() if edge[0] in active_node_set and edge[1] in active_node_set}
        # Each filtration gives 2 features: (number of active edges, number of active nodes)
        active_data.extend([len(active_node_set), len(active_edges)])  # 2 features per threshold
    
    return active_data  # 20-dimensional vector

# Process the graphs and generate embeddings (each embedding is 20-dimensional)
def process_graphs_for_embeddings(graphs):
    all_embeddings = []
    
    # Compute global thresholds based on all graphs
    thresholds = compute_degree_thresholds(graphs, num_thresholds=10)
    
    for graph in graphs:
        metric = dict(graph.degree())  # Degree centrality as metric
        active_data = graph_filtration(graph, metric, degree_activation, thresholds)   
        all_embeddings.append(active_data)  # 20-dimensional embedding
    
    return all_embeddings

# Parameters for graph generation (example from your previous code)
num_graphs = 1000  # Adjust the number of graphs for training
max_nodes = 100  # Maximum number of nodes
avg_edges_per_node = 5  # Average number of edges per node
period_days = 30  # Set the period of the cosine cycle
start_size = 10  # Starting size for nodes

# Generate synthetic graphs with cosine patterns for nodes and random edges
# synthetic_graphs = generate_cosine_pattern_graphs(num_graphs, max_nodes, avg_edges_per_node, period_days, start_size)

# Generate embeddings from synthetic graphs using learned thresholds
#embeddings = process_graphs_for_embeddings(synthetic_graphs)

In [ ]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.hidden_dim = hidden_dim
        self.Wa = nn.Linear(hidden_dim, hidden_dim)  # Attention weight matrix
        self.Ua = nn.Linear(hidden_dim, hidden_dim)  # Context weight matrix

    def forward(self, x):
        # Compute attention scores
        scores = torch.tanh(self.Wa(x))  # Shape: (batch_size, seq_length, hidden_dim)
        weights = torch.softmax(scores, dim=1)  # Shape: (batch_size, seq_length, hidden_dim)

        # Compute context vector as the weighted sum of the input
        context = torch.bmm(weights.transpose(1, 2), x)  # Shape: (batch_size, hidden_dim, hidden_dim)
        return context, weights

In [ ]:
class LSTMGRUPredictor(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim_1=64, hidden_dim_2=32, num_layers_LSTM=1, num_layers_GRU=1):
        super(LSTMGRUPredictor, self).__init__()
        self.hidden_dim_1 = hidden_dim_1
        self.hidden_dim_2 = hidden_dim_2
        self.hidden_dim_low_scaled = (self.hidden_dim_2//2)
        self.num_layers = num_layers
        
         # Define LSTM layers
        self.lstm1 = nn.LSTM(input_size=input_dim, hidden_size=self.hidden_dim_1, num_layers=num_layers_LSTM)
        self.lstm2 = nn.LSTM(input_size=self.hidden_dim_1, hidden_size=self.hidden_dim_2, num_layers=num_layers_LSTM)
        self.lstm3 = nn.LSTM(input_size=self.hidden_dim_2, hidden_size=self.hidden_dim_low_scaled, num_layers=num_layers_LSTM)

        # Define GRU layers
        self.gru1 = nn.GRU(input_size=self.hidden_dim_low_scaled, hidden_size=self.hidden_dim_low_scaled, num_layers=num_layers_GRU)
        self.gru2 = nn.GRU(input_size=self.hidden_dim_low_scaled, hidden_size=self.hidden_dim_low_scaled, num_layers=num_layers_GRU)
        self.gru3 = nn.GRU(input_size=self.hidden_dim_low_scaled, hidden_size=self.hidden_dim_low_scaled, num_layers=num_layers_GRU)

        # Fully connected layers
        self.fc1 = nn.Linear(self.hidden_dim_low_scaled, 100)
        self.fc2 = nn.Linear(100, output_dim)

        # Activation function
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):

        # Forward pass through the LSTM layers
        x, _ = self.lstm1(x)
        x, _ = self.lstm2(x)
        x, _ = self.lstm3(x)

        # Forward pass through the GRU layers
        x, _ = self.gru1(x)
        x, _ = self.gru2(x)
        x, _ = self.gru3(x)

        # Pass through the fully connected layers
        x = x[:, -1, :]  # Take the output from the last time step
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        
        return x

In [ ]:
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

# Assuming 'embeddings' is the full data you are working with
# Split the embeddings into training and test sets (80% train, 20% test)
#train_embeddings, test_embeddings = train_test_split(embeddings, test_size=0.2, shuffle=False)

# Dataset class for embeddings with flexibility for both train and test sets
class EmbeddingDataset(Dataset):
    def __init__(self, embeddings):
        self.embeddings = embeddings
    
    def __len__(self):
        return len(self.embeddings) - 1  # We want to predict the next embedding
    
    def __getitem__(self, idx):
        x = torch.Tensor(self.embeddings[idx]).unsqueeze(0)  # Add sequence dimension
        y = torch.Tensor(self.embeddings[idx + 1])  # Next embedding as target
        return x, y


def train_model(model, train_loader, optimizer, num_layers, hidden_dim_1, hidden_dim_2, criterion):
    model.train()
    epoch_loss = 0
    for x, y in train_loader:
        optimizer.zero_grad()

        # Reset hidden state for each batch (match batch size of x)
        hidden = ((torch.zeros(num_layers, x.size(0), hidden_dim_1).to(x.device), 
                    torch.zeros(num_layers, x.size(0), hidden_dim_1).to(x.device)),  
                    (torch.zeros(num_layers, x.size(0), hidden_dim_2).to(x.device),  
                    torch.zeros(num_layers, x.size(0), hidden_dim_2).to(x.device)),  
                    (torch.zeros(num_layers, x.size(0), hidden_dim_2).to(x.device), 
                    torch.zeros(num_layers, x.size(0), hidden_dim_2).to(x.device))
        )

        output = model(x)


        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(train_loader)
    


In [ ]:
def compute_overshoots(pred, real):
    overshoots = []
    for pred_val, real_val in zip(pred, real):
        overshoots.append(pred_val - real_val)

    return overshoots

# Compare Embeddings to GraphPulse Results

In [ ]:
import pandas as pd
import os
# Get GraphPulse Data
from my_utils.dataloader import MyDataLoader

my_loader = MyDataLoader()

datasets = ['networkadex', 'networkaragon', 'networkcoindash', 'mathoverflow', 'CollegeMsg']
num_layer = 3
hidden_dims = [64, 128, 256, 512]
dropouts = [0.2, 0.35, 0.5]
hidden_dim_1 = [64, 128, 256, 512]
hidden_dim_2 = [32, 64, 128, 256]
learning_rates = [0.0001, 0.001]
epochs = [500, 750, 1000]

dropout = 0.2

csv_file_path = 'results/Unbounded/graphpulsetesting_3layers.csv'

# Write the header if the file doesn't already exist
if not os.path.isfile(csv_file_path):
    pd.DataFrame(columns=['dataset', 'hidden_dim_1', 'hidden_dim_2', 'dropout', 'learning_rate', 'num_epochs', 'num_layers', 'aucroc', 'loss']).to_csv(csv_file_path, index=False)

for dataset in datasets:
    # Load and prep data
    data = my_loader.load_data(dataset)
    embeddings = process_graphs_for_embeddings(data)
    train_embeddings, test_embeddings = train_test_split(embeddings, test_size=0.2, shuffle=False)
    train_dataset = EmbeddingDataset(train_embeddings)
    test_dataset = EmbeddingDataset(test_embeddings)
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False)  
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
    
    #for hidden_dim in hidden_dims:
    #for dropout in dropouts:
    for hidden_1 in hidden_dim_1:
        for hidden_2 in hidden_dim_2:
            for lr_val in learning_rates:
                for epoch_val in epochs:
                    dropout_rate = dropout
                    input_dim = 20  # 20-dimensional embeddings
                    output_dim = 20  # Predict the next 20-dimensional embedding
                    num_layers = num_layer
                    learning_rate = lr_val

                    # Define the model (assuming LSTMGRUPredictor is defined elsewhere)
                    model = LSTMGRUPredictor(input_dim, output_dim, hidden_dim_1=hidden_1, hidden_dim_2=hidden_2, num_layers_LSTM=num_layers, num_layers_GRU=num_layers)
                    criterion = nn.MSELoss()
                    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
                    
                    for epoch in range(epoch_val):
                        loss = train_model(model, train_loader, optimizer, num_layers, hidden_1, hidden_2, criterion)
                        if (epoch + 1) % 50 == 0:
                            print(f"Epoch {epoch+1}/{epoch_val}, Loss: {loss}")

                    split_index = len(train_embeddings)  # The start of the test set time index
                    
                    model.eval()
                    test_loss = 0
                    time_index = split_index  # Start time index at the beginning of the test set

                    predicted_embeddings = []
                    real_embeddings = []
                    total_overshoots = []

                    with torch.no_grad():
                        hidden = None  # Initialize hidden state
                        for x, y in test_loader:
                            output = model(x)  # Maintain hidden state across time steps
                            loss = criterion(output, y)
                            test_loss += loss.item()
                            
                            # Print time index, predicted embedding, and real embedding
                            for i in range(len(x)):
                                predicted_embedding = output[i].numpy()
                                real_embedding = y[i].numpy()                            
                                predicted_embeddings.append(predicted_embedding)  # add to list for reconstruction
                                real_embeddings.append(real_embedding)
                                print(f"Time Index:\t{time_index}\tPredicted Embedding:\t{'\t'.join(map(str, predicted_embedding))}\tReal Embedding:\t{'\t'.join(map(str, real_embedding))}")
                                overshoots = compute_overshoots(predicted_embedding, real_embedding)
                                print(f'Time Index:\t{time_index}\tOvershot by (for each index) {overshoots}')  # Need to figure out how to store this
                                total_overshoots.append(overshoots)
                                # Plot the predicted vs real embeddings
                                plt.figure(figsize=(10, 6))
                                plt.plot(predicted_embedding, label='Predicted Embedding', marker='o')
                                plt.plot(real_embedding, label='Real Embedding', marker='x')
                                
                                plt.title('Predicted vs Real Embedding')
                                plt.xlabel('Embedding Dimension')
                                plt.ylabel('Value')
                                plt.legend()
                                
                                plt.grid(True)
                                plt.show()
                                
                                print("-" * 50)
                                time_index += 1

                    loss = test_loss/len(test_loader)
                    print(f"Test Loss: {loss}")
                    
                    # Write to dataframe
                    new_row = {
                        'dataset': dataset,
                        'hidden_dim_1': hidden_1,
                        'hidden_dim_2': hidden_2,
                        #'hidden_dim': hidden_dim,
                        'learning_rate': lr_val,
                        #'dropout': dropout,
                        'num_epochs': epoch_val,
                        'num_layers': num_layer,
                        #'aucroc': auc_roc, 
                        'loss': loss
                    }

                    # Calculate some proposed overshoots to test with
                    from statistics import mean
                    proposed_penalties = [mean(values) for values in zip(*total_overshoots)]
                    penalty_dict = {f'mean_overshoot_{i}': proposed_penalties[i] for i in range(len(proposed_penalties))}

                    new_row.update(penalty_dict)

                    pd.DataFrame([new_row]).to_csv(csv_file_path, mode='a', header=False, index=False)

In [ ]:
# Compare results to each other

# Reconstructing Graph (Three Methods)

In [ ]:
import time
import networkx as nx
import numpy as np
import math
import random

from my_utils.utils import Utils

Code used in all methods    

In [ ]:
# The real graphs to compare to
true_graphs = synthetic_graphs[(int(len(synthetic_graphs) * (0.8)) + 1):]  # Can change based on train/test split size

helper = Utils()

Helper Functions

In [ ]:

def greedy_fill(features):
    """
    Creates a graph such that we fill the graph using a greedy-inspired algorithm

    Args:
        features (list): The provided features for a graph (assuming only node_num and edge_num at each filtration)

    Returns:
        graph (nx.Graph()): The newly created graph
    """
    graph = nx.Graph()
    
    # Base graph features
    num_nodes = 0  # Also works to label nodes for adding edges
    num_edges = 0  
    curr_degree = 1  # Allows quick referencing of node degree
            
    for i in range(0, len(features), 2):
        total_num_nodes = features[i]  # Stores the number of nodes supposed to be in the graph now
        num_nodes_to_add = total_num_nodes - num_nodes
        for j in range(num_nodes_to_add):
            graph.add_node(num_nodes, degree=curr_degree)
            num_nodes += 1
        
        curr_degree += 1  # Increment the degree        
    
    num_edges_to_add = features[-1] - num_edges
    
    while num_edges < num_edges_to_add:
        edges_added = False
    
        unfilled_nodes = [node for node in graph.nodes if graph.degree(node) < graph.nodes[node]['degree']]  # Select all nodes that are not 'complete'
        random.shuffle(unfilled_nodes)
        
        for source in unfilled_nodes:
            if graph.degree(source) >= graph.nodes[source]['degree']:
                continue
            
            for target in unfilled_nodes:
                # Assuming no self loops are allowed
                if source == target:
                    continue
        
                if graph.degree(source) >= graph.nodes[source]['degree'] or graph.degree(target) >= graph.nodes[target]['degree']:
                    continue
        
                # Check if source and target dont have an edge
                if not graph.has_edge(source,target):
                    #print(f'Adding an edge to a node of degree {graph.degree(target)} that needs degree {graph.nodes[target]['degree']}')
                    graph.add_edge(source, target)
                    
                    # Updates
                    edges_added = True
                    num_edges += 1
                    
                    # Halt condition
                    if graph.degree(source) >= graph.nodes[source]['degree']:
                        break
                    
                    # Halt condition
                    if edges_added >= num_edges_to_add:
                        break
                    
                    break
            
            if edges_added:
                break
        
        if not edges_added:
            print('Cant add any more edges')
            break
            
    # Verify the graph was constructed properly
    unfilled_nodes = [node for node in graph.nodes if graph.nodes[node]['degree'] != graph.degree(node)]    
    if(len(unfilled_nodes)) > 0:
        for node in unfilled_nodes:
            print(f'Node #{node} is missing {(graph.nodes[node]['degree'] - graph.degree(node))} edges. Only has {graph.degree(node)} edges')
    
    return graph    

In [ ]:
predicted_embeddings = helper.round_features(predicted_embeddings, num_extra_features=0)
edit_distances = []
similarity_scores = [] 
graph_num = 1

# Create a graph and compare it to the corresponding true graph
for graph_embedding, true_graph in zip(predicted_embeddings, true_graphs):
    curr_graph = greedy_fill(graph_embedding)  # Create graph
    scores, edit_distance = helper.compute_similarity(curr_graph, true_graph)  # Get metrics
    
    # Add scores to the list
    similarity_scores.append(scores)
    edit_distances.append(edit_distance)
    
    graph_num += 1
    
    
# Display how far off we were for each metric as plt
helper.display_edit_differences(edit_distances)

Method 2: Local Search Based on Node Degree Completiton

In [ ]:
def local_assemble(features):
    graph = nx.Graph()

    num_nodes = 0  # Also works to label nodes for adding edges
    num_edges = 0  
    curr_degree = 1  # Allows quick referencing of node degree
            
    for i in range(0, len(features), 2):
        total_num_nodes = features[i]  # Stores the number of nodes supposed to be in the graph now
        num_nodes_to_add = total_num_nodes - num_nodes
        for j in range(num_nodes_to_add):
            graph.add_node(num_nodes, degree=curr_degree)
            num_nodes += 1
        
        curr_degree += 1  # Increment the degree        
    
    # Step 2: Start local search for edge creation
    max_iterations = 1000
    iterations = 0
    improvements = True

    while iterations < max_iterations and improvements:
        improvements = False

        unfilled_nodes = [node for node in graph.nodes if graph.degree(node) < graph.nodes[node]['degree']]
        
        if len(unfilled_nodes) < 2:
            break

        # Randomly select a pair of nodes to attempt to connect
        source = random.choice(unfilled_nodes)
        target = random.choice(unfilled_nodes)

        if source != target and not graph.has_edge(source, target):
            # Check if adding this edge would help meet degree requirements
            if (graph.degree(source) < graph.nodes[source]['degree'] and graph.degree(target) < graph.nodes[target]['degree']):
                graph.add_edge(source, target)
                improvements = True

            # If adding the edge causes any node to exceed its degree, we might need to adjust
            for node in graph.nodes:
                if graph.degree(node) > graph.nodes[node]['degree']:
                    # Remove an edge to balance
                    edges = list(graph.edges(node))
                    if edges:
                        edge_to_remove = random.choice(edges)
                        graph.remove_edge(*edge_to_remove)

        iterations += 1

    # Verify the graph was constructed properly
    unfilled_nodes = [node for node in graph.nodes if graph.nodes[node]['degree'] != graph.degree(node)]    
    if(len(unfilled_nodes)) > 0:
        for node in unfilled_nodes:
            print(f'Node #{node} is missing {(graph.nodes[node]['degree'] - graph.degree(node))} edges. Only has {graph.degree(node)} edges')

    return graph

In [ ]:
def local_assemble_degree(features):
    graph = nx.Graph()

    num_nodes = 0  # Also works to label nodes for adding edges
    num_edges = 0  
    curr_degree = 1  # Allows quick referencing of node degree
    necessary_edges = helper.calculate_necessary_edges(features)
    
    print(f'There should be {necessary_edges} edges in this graph with feature vector {features}')  # Debugging  
    
    # Fill a graph of just nodes (no edges yet)
    for i in range(0, len(features), 2):
        total_num_nodes = features[i]  # Stores the number of nodes supposed to be in the graph now
        num_nodes_to_add = total_num_nodes - num_nodes
        for j in range(num_nodes_to_add):
            graph.add_node(num_nodes, degree=curr_degree)
            num_nodes += 1
        
        curr_degree += 1  # Increment the degree        
    
    max_iterations = 5000
    iterations = 0
    trials_without_improvements = 0
    improvements_limit = 500
    curr_unfilled = num_nodes  # To track our improvements
    no_candidates = False

    while iterations < max_iterations and trials_without_improvements < improvements_limit:
        # Need to add a link to the graph
        if num_edges < necessary_edges and not no_candidates:
            unfilled_nodes = [node for node in graph.nodes if graph.degree(node) < graph.nodes[node]['degree']]
        
            if len(unfilled_nodes) < 2:
                no_candidates = True
                continue

            # Randomly select a pair of nodes to attempt to connect
            source = random.choice(unfilled_nodes)
            target = random.choice(unfilled_nodes)

            if source != target and not graph.has_edge(source, target):
                tmp_graph = graph.copy()
                tmp_graph.add_edge(source, target)
                new_unfilled = helper.count_incomplete(tmp_graph)

                # Need to think of a way to represent there wasnt an improvement but we also didnt worsen (thinking of doing candidates)
                if new_unfilled < curr_unfilled:
                    graph = tmp_graph.copy()
                    trials_without_improvements = 0
                
                else:
                    trials_without_improvements += 1

                num_edges += 1

        # Delete an edge and try it somewhere else
        elif num_edges == necessary_edges or no_candidates:
            # Choose a random edge to delete
            edge = random.choice(list(graph.edges()))
            source, target = edge

            tmp_graph = graph.copy()

            tmp_graph.remove_edge(source, target)
            num_edges -= 1

            # Below is same logic of adding edge normally
            unfilled_nodes = [node for node in graph.nodes if graph.degree(node) < graph.nodes[node]['degree']]
        
            if len(unfilled_nodes) < 2:
                no_candidates = False 
                num_edges -= 1
                continue

            # Randomly select a new pair of nodes to attempt to connect
            source = random.choice(unfilled_nodes)
            target = random.choice(unfilled_nodes)

            if source != target and not graph.has_edge(source, target):
                tmp_graph.add_edge(source, target)
                new_unfilled = helper.count_incomplete(tmp_graph)

                # Need to think of a way to represent there wasnt an improvement but we also didnt worsen (thinking of doing candidates)
                if new_unfilled < curr_unfilled:
                    graph = tmp_graph.copy()
                    trials_without_improvements = 0
                
                else:
                    trials_without_improvements += 1

                num_edges += 1


        # Debugging, we should never have this occur
        elif num_edges > necessary_edges:
            print('TOO MANY EDGES')

        iterations += 1

    # Verify the graph was constructed properly
    unfilled_nodes = [node for node in graph.nodes if graph.nodes[node]['degree'] != graph.degree(node)]    
    if(len(unfilled_nodes)) > 0:
        for node in unfilled_nodes:
            print(f'Node #{node} is missing {(graph.nodes[node]['degree'] - graph.degree(node))} edges. Only has {graph.degree(node)} edges')

    return graph

In [ ]:
edit_distances = []
similarity_scores = [] 

# Create a graph and compare it to the corresponding true graph
for graph_embedding, true_graph in zip(predicted_embeddings, true_graphs):
    curr_graph = local_assemble_degree(graph_embedding)  # Create graph
    scores, edit_distance = helper.compute_similarity(curr_graph, true_graph)  # Get metrics
    
    # Add scores to the list
    similarity_scores.append(scores)
    edit_distances.append(edit_distance)
    
    
# Display how far off we were for each metric as plt
helper.display_edit_differences(edit_distances)
helper.display_similarity_values(similarity_scores)

Method 3: Local Search Based on Degree (Genetics)

In [ ]:
def local_assemble_genetics(features, num_candidates):
    graph = nx.Graph()



    return graph

# Self Notes

- Compare Embeddings to GraphPulse Growth/Shrink metric
- Implement Linear Regression on the embeddings
- Allow for multiedges
- Look at the concept of rewiring, call my greedy wiring
- Try the step 1 one again but try to impute the average/max or smth when we don't have monotonic behavior
- I need to make a probability function to determine what nodes will receive the next edge
- Include assertivity in the LSTMGRU Embedding for prediction
- Look at GAEs, they create a graph embedding and try to reconstruct the graph from the embedding, I need to look at how they reconstruct the graph
- They probably use some learnable parameters